In [1]:
import os
import math
import matplotlib.pyplot as plt

import verde as vd
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box


from typing import Tuple

In [2]:
gdf = pd.read_csv('../data/labels/dataset_v5/labels.csv')
gdf['geometry'] = gpd.GeoSeries.from_wkt(gdf['geometry'])
gdf = gpd.GeoDataFrame(gdf)
gdf.crs = 4326



In [3]:
#gdf.loc[gdf.driver == 'soy'].country.value_counts()
#gdf.loc[gdf.country == 'Bolivia'].driver.value_counts()

In [4]:

'''
Potential countries for testing


------------------------------------
Malaysia
driver
palm oil           339
rubber             259
urban              221
forest regrowth     58
mining              19

Tabular:
- cattle 6.1, forest plant, 3.45, rubber 15.7, oil palm 45.3
------------------------------------

Paraguay

pasture            404
soy                187
forest regrowth     16
mining               5
urban                3

Tabular:
- cattle 80, soya 7.1

------------------------------------
Colombia

pasture            300
forest regrowth    126
urban               58
palm oil            58
mining              52
cocoa               15
other                4
coffee               1

Tabular
- Cattle 64, cocoa 1.9, coffee 2.3, forest 3.9, palm oil 5.3


------------------------------------

Papua New Guinea
driver
palm oil           281
forest regrowth     57
cocoa               48
mining              26


Tabular:
- Cattle 15, Palm oil 24, cocoa 12
------------------------------------
Costa Rica
palm oil           217
coffee              40
urban               26
forest regrowth      4
mining               1

Palm oil 12.5, Coffee 11
------------------------------------
Ghana
cocoa                   1024
mining                   132
urban                     29
palm oil                  24
forest plantation         12
rubber                    12
shifting agriculture       2
forest regrowth            2

Tabular
- cattle 13, cocoa 30, oil palm 3.7
------------------------------------

Ethiopia
coffee             107
forest regrowth     24
urban               15
mining               4

- cattle 3.8, coffee 7,4


------------------------------------------------------------------------
------------------------------------------------------------------------
need to select those countries that are small enough so they fit in the splits 
- not colombia
- ghana, costa rica, paraguay, malaysia, papua


'''





'\nPotential countries for testing\n\n\n------------------------------------\nMalaysia\ndriver\npalm oil           339\nrubber             259\nurban              221\nforest regrowth     58\nmining              19\n\nTabular:\n- cattle 6.1, forest plant, 3.45, rubber 15.7, oil palm 45.3\n------------------------------------\n\nParaguay\n\npasture            404\nsoy                187\nforest regrowth     16\nmining               5\nurban                3\n\nTabular:\n- cattle 80, soya 7.1\n\n------------------------------------\nColombia\n\npasture            300\nforest regrowth    126\nurban               58\npalm oil            58\nmining              52\ncocoa               15\nother                4\ncoffee               1\n\nTabular\n- Cattle 64, cocoa 1.9, coffee 2.3, forest 3.9, palm oil 5.3\n\n\n------------------------------------\n\nPapua New Guinea\ndriver\npalm oil           281\nforest regrowth     57\ncocoa               48\nmining              26\n\n\nTabular:\n- Catt

In [5]:
## Paragua, Ghana, Costa Rica, Papua/Malaysia

## Goals
- have one test country for each continent
- have blocks that cover entire test countries -> prevent leakage

- similar scenario (countries within blocks) should be there for train/val

### Implementation

- divide into blocks manually
- test various block sizes
- try out some shifts to make the test countries fall entirely within blocks



In [6]:


def create_grid(square: Polygon, width: float, height: float) -> gpd.GeoDataFrame:

    minx, miny, maxx, maxy = square.bounds

    ncols = int((maxx - minx) / width)
    nrows = int((maxy - miny) / height)
    
    grid_cells = []
    for i in range(nrows):
        for j in range(ncols):
            cell_minx = minx + j * width
            cell_miny = miny + i * height
            cell_maxx = cell_minx + width
            cell_maxy = cell_miny + height
            grid_cells.append(box(cell_minx, cell_miny, cell_maxx, cell_maxy))
    
    grid_gdf = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
    return grid_gdf

#square_polygon = box(*gdf.total_bounds)  
cell_width, cell_height = 5, 5



In [23]:
#Americas
#These blocks look good
# 10 deg size
#  both Paraguay and Costa Rica in a single block
# Ecuador also -> validation? ; also Suriname and Guayana
min_x, min_y = (-112, -38)
max_x, max_y = (-32, 32)

bbox_coords = [(min_x, min_y), (min_x, max_y), (max_x, max_y), (max_x, min_y), (min_x, min_y)]

# Create the Polygon
bbox_polygon = Polygon(bbox_coords)

grid_america = create_grid(bbox_polygon, cell_width, cell_height)
grid_america.to_file('../data/tmp/america_grid_5.gpkg')

In [46]:
#Should be only one 

#gdf.loc[gdf.country == 'Costa Rica'].total_bounds
intersect_using_spatial_index(grid_america, gdf.loc[gdf.country == 'Costa Rica'])

,geometry
149,"POLYGON ((-82.00000 7.00000, -82.00000 12.0000..."


In [24]:
#

In [47]:
#Africa 
#If both Cote d'Ivoire and Ghana are in test,
# - the overall accuracy may be lower because it's more difficult
# - the loc may add less 
# - the tab data may add more

# If Cote d'Ivoire is in train/val
# - the location will add a lot, and overall it should have high cocoa accuracy
# - the tab data may not result in an improvement
# let's go with this option for now, 


### This way, Ghana is in one split -> test
# Ethiopia is over two blocks, can be made into test/val





min_x, min_y = (-24, -25.5)
max_x, max_y = (56, 24.5)

bbox_coords = [(min_x, min_y), (min_x, max_y), (max_x, max_y), (max_x, min_y), (min_x, min_y)]

# Create the Polygon
bbox_polygon = Polygon(bbox_coords)

grid_africa = create_grid(bbox_polygon, cell_width, cell_height)


grid_africa.to_file('../data/tmp/africa_grid_5.gpkg')

In [48]:
#Should be only one 
intersect_using_spatial_index(grid_africa, gdf.loc[gdf.country == 'Ghana'])

,geometry
100,"POLYGON ((1.00000 4.50000, 1.00000 9.50000, -4..."


### Asia

In [59]:
#Asia 
# one split -> Cambodia

change_country_to_thailand_ids = [20008376, 20009420, 20009417, 20009028, 20008953, 20008889]

gdf.loc[gdf.sampleid.isin(change_country_to_thailand_ids), 'country'] = 'Thailand'
pd.DataFrame(gdf).to_csv('../data/labels/dataset_v5/labels.csv', index=False)

#gdf.loc[gdf.country == 'Cambodia'].total_bounds
#cambodia_cells = 



In [44]:


min_x, min_y = (72.5, -30)
max_x, max_y = (172.5, 30)

bbox_coords = [(min_x, min_y), (min_x, max_y), (max_x, max_y), (max_x, min_y), (min_x, min_y)]

# Create the Polygon
bbox_polygon = Polygon(bbox_coords)

grid_asia = create_grid(bbox_polygon, cell_width, cell_height)
grid_asia.to_file('../data/tmp/asia_grid_5.gpkg')

In [58]:
#Should be only one 
intersect_using_spatial_index(grid_asia, gdf.loc[gdf.country == 'Cambodia'])

,geometry
165,"POLYGON ((102.50000 10.00000, 102.50000 15.000..."
166,"POLYGON ((107.50000 10.00000, 107.50000 15.000..."


In [49]:

grid = pd.concat([grid_america, grid_africa, grid_asia]).reset_index(drop=True)
grid['_id'] = grid.index

## Discard empty grid cells

In [131]:

labels = pd.read_csv('../data/labels/dataset_v5/labels.csv')
labels['geometry'] = gpd.GeoSeries.from_wkt(labels['geometry'])
labels = gpd.GeoDataFrame(labels)
labels.crs = 4326


In [69]:
#gdf.to_file('../data/tmp/labels_v5.gpkg')

In [132]:
def intersect_using_spatial_index(source_gdf, intersecting_gdf):
    """
    Conduct spatial intersection using spatial index for candidates GeoDataFrame to make queries faster.
    Note, with this function, you can have multiple Polygons in the 'intersecting_gdf' and it will return all the points 
    intersect with ANY of those geometries.
    """
    source_sindex = source_gdf.sindex
    possible_matches_index = []
    
    # 'itertuples()' function is a faster version of 'iterrows()'
    for other in intersecting_gdf.itertuples():
        bounds = other.geometry.bounds
        c = list(source_sindex.intersection(bounds))
        possible_matches_index += c
    
    # Get unique candidates
    unique_candidate_matches = list(set(possible_matches_index))
    possible_matches = source_gdf.iloc[unique_candidate_matches]

    # Conduct the actual intersect
    result = possible_matches.loc[possible_matches.intersects(intersecting_gdf.unary_union)]
    return result

In [135]:
grid = intersect_using_spatial_index(grid, gdf).reset_index(drop=True)
grid['_id'] = grid.index
grid.crs = 4326

In [188]:
grid.to_file('../data/tmp/grid_5.gpkg')

### Dividing cells into splits

### 1. manual

In [14]:
#TODO: find the cell ids to be in the test set


#let's try smaller countries

#Paraguay/Costa Rica

#Ghana

# Cambodia


In [163]:
test_countries = ['Cambodia', 'Costa Rica', 'Ghana']
test_cells = intersect_using_spatial_index(grid, gdf.loc[gdf.country.isin(test_countries) ])._id.to_list()

### 2. random

- based on this, maybe divide the high-density and low-density separately to have some equality between splits

In [138]:
labels = labels.drop(['split_block','split'], axis=1)

In [139]:
labels = gpd.sjoin(labels, grid, how='left')

In [141]:
labels = labels.drop('_id', axis=1)

In [142]:

labels = labels.rename(columns={'index_right' : 'block_id'})

In [145]:
labels['split'] = None

In [165]:
labels.loc[labels.block_id.isin(test_cells), 'split'] = 4

In [166]:
labels

,driver,sampleid,dataset,country,geometry,block_id,split
0,cocoa,170000000,cocoa_peru_points.csv,Peru,POINT (-78.18009 -4.44482),66.0,None
1,cocoa,170000002,cocoa_peru_points.csv,Peru,POINT (-76.32139 -6.30353),67.0,None
2,cocoa,170000003,cocoa_peru_points.csv,Peru,POINT (-76.11684 -6.31242),67.0,None
3,cocoa,170000004,cocoa_peru_points.csv,Peru,POINT (-75.77000 -6.34799),67.0,None
4,cocoa,170000005,cocoa_peru_points.csv,Peru,POINT (-76.05459 -6.36578),67.0,None
...,...,...,...,...,...,...,...
33601,forest plantation,10698584,mapbiomas.csv,Brazil,POINT (-50.31073 -30.15563),14.0,None
33602,forest plantation,80063592,mapbiomas_indonesia.csv,Indonesia,POINT (102.87699 0.63749),202.0,None
33603,forest plantation,10686066,mapbiomas.csv,Brazil,POINT (-49.31077 -22.11216),32.0,None
33604,forest plantation,10649958,mapbiomas.csv,Brazil,POINT (-41.58247 -15.75843),46.0,None


In [151]:
import random 
import copy

def partition (list_in, n, seed):
    
    random.Random(seed).shuffle(list_in)
    return [list_in[i::n] for i in range(n)]



def test_random_split(indices, gdf, seed):
    
    drivers = ['cocoa', 'coffee', 'rubber', 'shifting agriculture',
       'forest plantation', 'palm oil', 'other', 'urban', 'mining',
       'forest regrowth', 'soy', 'pasture']
    _indices = copy.deepcopy(indices)
    split_0, split_1, split_2, split_3 = partition(_indices, 4, seed)
    
    for split in [split_0, split_1, split_2, split_3]:
        split_data = gdf.loc[gdf.block_id.isin(split)]
        class_dist = split_data.driver.value_counts()
        
        print("dst size: ", len(split_data))
        print('classes in split: ', len(class_dist))
        print(class_dist) 
    

In [167]:
train_val_blocks = [i for i in grid._id.unique() if i not in test_cells]



In [177]:
seed = 9

test_random_split(train_val_blocks, labels, seed)

dst size:  7750
classes in split:  12
driver
forest plantation       996
forest regrowth         956
pasture                 904
mining                  835
rubber                  701
palm oil                640
other                   564
soy                     561
urban                   519
shifting agriculture    461
coffee                  366
cocoa                   247
Name: count, dtype: int64
dst size:  7290
classes in split:  12
driver
rubber                  1091
cocoa                   1091
mining                   810
other                    625
pasture                  609
soy                      603
forest regrowth          595
urban                    570
shifting agriculture     559
palm oil                 374
forest plantation        235
coffee                   128
Name: count, dtype: int64
dst size:  9880
classes in split:  12
driver
soy                     1258
shifting agriculture    1139
forest plantation       1052
other                   1005
urban        

In [191]:
seed = 9
split_0, split_1, split_2, split_3 = partition(copy.deepcopy(train_val_blocks), 4, seed)

In [261]:
data_split_mapping = {
    0 : split_0,
    1 : split_1,
    2 : split_2,
    3 : split_3,
    4 : test_cells
}
data_split_mapping = {value: key for key, values in data_split_mapping.items() for value in values}


In [262]:
#Manually adding some more blocks to test set
data_split_mapping[134] = 4
#data_split_mapping[58] = 4 #TODO: replace with another one 
data_split_mapping[56] = 4 # <- replaced 58
data_split_mapping[69] = 4
data_split_mapping[203] = 4
data_split_mapping[43] = 4
data_split_mapping[192] = 4
data_split_mapping[154] = 4
#data_split_mapping[32] = 4
data_split_mapping[33] = 4
data_split_mapping[71] = 4

In [263]:
labels['split'] = labels.block_id.map(data_split_mapping)

In [264]:
#Drop one broken rows
#labels = labels.drop(5806, axis=0)


In [265]:
labels.split.value_counts()

split
2    8832
0    7342
3    6513
1    5841
4    5077
Name: count, dtype: int64

In [266]:
labels.driver.value_counts()

driver
rubber                  3013
other                   2994
mining                  2994
cocoa                   2993
soy                     2991
forest plantation       2990
pasture                 2989
palm oil                2984
shifting agriculture    2978
forest regrowth         2975
urban                   2931
coffee                   773
Name: count, dtype: int64

In [267]:
labels.loc[labels.split == 4].driver.value_counts()

driver
cocoa                   1128
mining                   532
rubber                   470
soy                      444
other                    429
pasture                  422
palm oil                 418
shifting agriculture     293
forest regrowth          263
urban                    259
forest plantation        221
coffee                   198
Name: count, dtype: int64

In [268]:
labels

,driver,sampleid,dataset,country,geometry,block_id,split
0,cocoa,170000000,cocoa_peru_points.csv,Peru,POINT (-78.18009 -4.44482),66.0,0
1,cocoa,170000002,cocoa_peru_points.csv,Peru,POINT (-76.32139 -6.30353),67.0,3
2,cocoa,170000003,cocoa_peru_points.csv,Peru,POINT (-76.11684 -6.31242),67.0,3
3,cocoa,170000004,cocoa_peru_points.csv,Peru,POINT (-75.77000 -6.34799),67.0,3
4,cocoa,170000005,cocoa_peru_points.csv,Peru,POINT (-76.05459 -6.36578),67.0,3
...,...,...,...,...,...,...,...
33601,forest plantation,10698584,mapbiomas.csv,Brazil,POINT (-50.31073 -30.15563),14.0,0
33602,forest plantation,80063592,mapbiomas_indonesia.csv,Indonesia,POINT (102.87699 0.63749),202.0,3
33603,forest plantation,10686066,mapbiomas.csv,Brazil,POINT (-49.31077 -22.11216),32.0,3
33604,forest plantation,10649958,mapbiomas.csv,Brazil,POINT (-41.58247 -15.75843),46.0,0


In [269]:
labels.to_file('../data/tmp/test_split_v2.gpkg')

In [272]:
labels.split.value_counts()

split
2    8832
0    7342
3    6513
1    5841
4    5077
Name: count, dtype: int64

In [271]:
pd.DataFrame(labels).to_csv('../data/labels/dataset_v5/labels_grid_5_split.csv', index=False)

### Manual 

In [ ]:
test_cells

In [824]:
test_countries = ['Ghana', 'Costa Rica', 'Paraguay', 'Malaysia','Colombia', 'Papua New Guinea']
world['split'] = 'train'
world.loc[world.name.isin(test_countries), 'split'] = 'test'

In [165]:
gdf.loc[gdf.country.isin(test_countries)].driver.unique()

array(['cocoa', 'coffee', 'rubber', 'shifting agriculture',
       'forest plantation', 'palm oil', 'other', 'urban', 'mining',
       'forest regrowth', 'soy', 'pasture'], dtype=object)

In [828]:
gdf['split'] = 'train'
gdf.loc[gdf.country.isin(test_countries), 'split'] = 'test'

In [829]:
gdf.to_file('../data/tmp/labels_v5.gpkg')

In [830]:
gdf.loc[gdf.country == 'Ethiopia'].driver.value_counts()

driver
coffee             107
forest regrowth     24
urban               15
mining               4
Name: count, dtype: int64

/tmp/ipykernel_2854489/773290482.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tropical_geometry_buffer['geometry'] = tropical_geometry_buffer['geometry'].buffer(0.01)


In [854]:
tropical_geometry_buffer = tropical_countries.copy(deep=True)
tropical_geometry_buffer['geometry'] = tropical_geometry_buffer['geometry'].buffer(1)
gdf = gpd.sjoin(gdf, tropical_geometry_buffer[['geometry', 'name']], how='left', predicate='within')[['geometry', 'driver', 'sampleid', 'dataset', 'name']]
gdf['country'] = gdf['name']
gdf = gdf.drop('name', axis=1)

/tmp/ipykernel_2854489/586653345.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  tropical_geometry_buffer['geometry'] = tropical_geometry_buffer['geometry'].buffer(1)


In [855]:
gdf.loc[gdf.country.isna()].to_file('../data/tmp/del.gpkg')

In [856]:
gdf.loc[gdf.country.isna()].driver.value_counts()

driver
urban              65
mining             33
rubber             33
forest regrowth    32
other              23
palm oil            5
Name: count, dtype: int64

In [811]:
for _country in gdf.country.value_counts().head(20).index:
    print(_country)
    print(gdf.loc[gdf.country == _country].driver.value_counts())
    print()

Brazil
driver
soy                  2662
pasture              2072
forest plantation    1770
other                 885
mining                780
forest regrowth       682
urban                 643
coffee                164
palm oil               34
Name: count, dtype: int64

Indonesia
driver
other                1579
forest plantation    1079
palm oil              945
forest regrowth       609
mining                580
urban                 523
rubber                 76
Name: count, dtype: int64

Dem. Rep. Congo
driver
shifting agriculture    2196
palm oil                 159
forest regrowth           96
urban                     84
mining                    25
Name: count, dtype: int64

Côte d'Ivoire
driver
cocoa                   1195
palm oil                  37
urban                     27
mining                    18
shifting agriculture      10
forest regrowth            6
rubber                     5
coffee                     1
Name: count, dtype: int64

Thailand
driver
rubber  

In [793]:
sort_by_column_ratio(per_country, 'Soya beans').head(15)

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans,Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),Forest plantation (Needleleaf),Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak)
country,,,,,,,,,,,,,,,,,,,,
Belize,3.340203,1.290443,0.043566,0.488799,0.175800,NaN,0.006985,15.852755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Rwanda,NaN,NaN,1.733621,0.658416,NaN,NaN,0.553183,4.339003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000244,NaN,NaN
Zambia,15.700090,NaN,0.069195,NaN,0.826321,NaN,NaN,13.276410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Benin,12.634286,NaN,0.000910,0.034094,0.664962,NaN,1.723134,7.433725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bolivia (Plurinational State of),47.307818,0.095735,0.165045,0.000029,2.489885,NaN,NaN,16.963033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
El Salvador,1.653853,0.498332,0.566315,4.250887,0.087045,NaN,NaN,1.097430,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Brazil,75.067694,0.347720,0.400348,1.962791,3.950931,0.094956,0.033149,9.018037,NaN,NaN,NaN,1.903619e-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethiopia,3.808997,NaN,7.429780,1.536680,0.200474,NaN,NaN,1.129331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Paraguay,79.855537,NaN,0.000301,0.478945,4.202923,NaN,0.009970,7.193152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [770]:
per_country

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans,Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),Forest plantation (Needleleaf),Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak)
country,,,,,,,,,,,,,,,,,,,,
Angola,28.713242,0.012401,0.883636,NaN,1.511223,NaN,0.013145,1.097869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Australia,13.507555,NaN,NaN,62.590021,0.710924,NaN,NaN,0.071757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bahamas,55.548494,NaN,NaN,NaN,2.923605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bangladesh,15.621705,NaN,NaN,11.943748,0.822195,2.409705,NaN,0.567638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Belize,3.340203,1.290443,0.043566,0.488799,0.175800,NaN,0.006985,15.852755,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
United States of America,6.418858,NaN,0.000355,82.170583,0.337835,NaN,NaN,3.040608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vanuatu,19.730972,24.193539,0.350354,NaN,1.038472,NaN,0.016324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Venezuela (Bolivarian Republic of),47.447586,1.519243,0.135711,17.698015,2.497241,NaN,1.147772,1.452792,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [771]:
per_country.div(per_country.sum(axis=1), axis=0)

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans,Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),Forest plantation (Needleleaf),Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak)
country,,,,,,,,,,,,,,,,,,,,
Angola,0.890844,0.000385,0.027415,NaN,0.046887,NaN,0.000408,0.034062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Australia,0.175696,NaN,NaN,0.814123,0.009247,NaN,NaN,0.000933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bahamas,0.950000,NaN,NaN,NaN,0.050000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Bangladesh,0.498062,NaN,NaN,0.380799,0.026214,0.076828,NaN,0.018098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Belize,0.157568,0.060874,0.002055,0.023058,0.008293,NaN,0.000330,0.747823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
United States of America,0.069794,NaN,0.000004,0.893467,0.003673,NaN,NaN,0.033062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vanuatu,0.435277,0.533724,0.007729,NaN,0.022909,NaN,0.000360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Venezuela (Bolivarian Republic of),0.659926,0.021130,0.001888,0.246153,0.034733,NaN,0.015964,0.020206,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
'''#Add country name
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])
tropical_countries = world.loc[world.intersects(tropics)]



gdf.crs = 4326
gdf = gpd.sjoin(gdf, tropical_countries[['geometry', 'name']], how='left', predicate='within')[['geometry', 'driver', 'sampleid', 'dataset', 'name']]
gdf['country'] = gdf['name']
gdf = gdf.drop('name', axis=1)'''

## OLD Sampling limited number of examples per class

 - first, do it just by simply randomly sampling
 - look at the results in QGIS and check spatial bias (e.g., if 

In [4]:
gdf = pd.read_csv('../data/labels/dataset_v4/labels.csv')

In [335]:
gdf = gdf_without_conflicts 

In [336]:
driver_classes = np.unique(gdf.driver, return_counts=True)[0]
max_count_per_class = 5000
def flatten(nested_list):
    return [x for xs in nested_list for x in xs]


indices = []
for _class in driver_classes:
    try:
        class_samples = gdf.loc[gdf.driver == _class].sample(max_count_per_class, random_state=1).index
    except ValueError:
        class_samples = gdf.loc[gdf.driver == _class].index
    indices.append(class_samples.tolist())

indices = flatten(indices)

subset = gdf.loc[indices]

In [337]:
cls, count = np.unique(subset.driver, return_counts=True)


for i in range(len(cls)):
    print(cls[i], count[i])

cocoa 5000
coffee 416
forest plantation 5000
forest regrowth 5000
mining 5000
other 5000
palm oil 5000
pasture 5000
rubber 5000
shifting agriculture 5000
soy 5000
urban 5000


In [338]:
subset['geometry'] = subset['geometry'].centroid

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


In [339]:
subset.to_file('../data/tmp/subset_5k_per_class.gpkg')
pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


## Testing  stratified sampling of examples

In [73]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split

# Load your dataset
# Assume data is a pandas DataFrame with columns: ['latitude', 'longitude', 'class']
# Replace with the actual way you load your data
data = pd.read_csv('../data/labels/dataset_v4/labels.csv')
data['geometry'] = gpd.GeoSeries.from_wkt(data['geometry'])
data = gpd.GeoDataFrame(data)
data['geometry'] = data.geometry.centroid
data['latitude'] = data.geometry.x
data['longitude'] = data.geometry.y


# Extract coordinates and class labels
coordinates = data[['latitude', 'longitude']]
labels = data['driver']

# Cluster the data using DBSCAN
db = DBSCAN(eps=0.1, min_samples=10).fit(coordinates)  # Adjust eps and min_samples based on your data
data['cluster'] = db.labels_

# Remove noise points if any (DBSCAN labels noise as -1)
data = data[data['cluster'] != -1]

# Function to perform stratified sampling within each cluster
def stratified_sample(data, n_samples_per_class):
    samples = []
    clusters = data['cluster'].unique()
    
    for cluster in clusters:
        cluster_data = data[data['cluster'] == cluster]
        for class_label in cluster_data['driver'].unique():
            class_data = cluster_data[cluster_data['driver'] == class_label]
            if len(class_data) >= n_samples_per_class:
                sample = class_data.sample(n_samples_per_class, random_state=42)
            else:
                sample = class_data  # if not enough samples, take all
            samples.append(sample)
    
    return pd.concat(samples)

# Sample 5000 samples per class
n_samples_per_class = 5000
sampled_data = stratified_sample(data, n_samples_per_class)

# Ensure balance across classes
final_sampled_data = []
for class_label in sampled_data['driver'].unique():
    class_samples = sampled_data[sampled_data['driver'] == class_label]
    if len(class_samples) > n_samples_per_class:
        final_sampled_data.append(class_samples.sample(n_samples_per_class, random_state=42))
    else:
        final_sampled_data.append(class_samples)

final_sampled_data = pd.concat(final_sampled_data)

# Shuffle the final sampled dataset
final_sampled_data = final_sampled_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Save the sampled dataset
final_sampled_data.to_csv('../data/tmp/sampled_dataset.csv', index=False)


In [76]:
final_sampled_data.to_file('../data/tmp/sampled_dataset.gpkg')

In [74]:
final_sampled_data

,geometry,driver,sampleid,dataset,latitude,longitude,cluster
0,POINT (109.32403 -0.05040),urban,100006906,worldcover.csv,109.324027,-0.050395,168
1,POINT (102.68012 2.56460),rubber,20036784,rubber.csv,102.680124,2.564600,156
2,POINT (102.10537 1.47439),urban,100007595,worldcover.csv,102.105372,1.474391,169
3,POINT (-50.40195 -24.87920),pasture,10641491,mapbiomas.csv,-50.401950,-24.879199,747
4,POINT (-48.25448 -2.42356),urban,10322019,mapbiomas.csv,-48.254484,-2.423565,297
...,...,...,...,...,...,...,...
55306,POINT (-42.09707 -22.65264),pasture,10655187,mapbiomas.csv,-42.097071,-22.652635,820
55307,POINT (17.13554 -5.19153),shifting agriculture,130001450,tyukavina_congo_basin_agri.csv,17.135538,-5.191531,1502
55308,POINT (-79.62891 -0.91170),cocoa,110002239,cocoa_ecuador.csv,-79.628906,-0.911700,968
55309,POINT (105.47272 14.76049),rubber,20010658,rubber.csv,105.472725,14.760488,127


In [79]:
data.loc[data.driver == 'palm oil'].to_file('../data/tmp/palmoil.gpkg')

## Block data split

In [5]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df)


In [6]:
gdf = gpd.GeoDataFrame(df)


In [7]:

coordinates = (gdf.geometry.x, gdf.geometry.y)


In [8]:
#spacing = 1
spacing = 5


block_coordinates, labels = vd.block_split(
    coordinates,
    spacing=spacing,
    #random_state=123,
)

In [9]:
blocks = np.unique(labels)

In [10]:
gdf['tmp'] = labels

In [39]:
gdf.to_file('../data/tmp/labels_55k.gpkg')

In [30]:
#Find such split that the each fold has at least n examples of all classes


finished = False
required_n_examples_per_class = 10

while not finished:
    # Shuffle the array
    np.random.shuffle(blocks)

    # Split the array into 5 equal parts
    split_arrays = np.array_split(blocks, 5)

    '''
    split 0-3 are for 4-fold cross validation (train-val)
    split 4 is test set 

    '''

    splits = []

    for item in labels:
        if item in split_arrays[0]:
            splits.append(0)
        elif item in split_arrays[1]:
            splits.append(1)
        elif item in split_arrays[2]:
            splits.append(2)
        elif item in split_arrays[3]:
            splits.append(3)
        elif item in split_arrays[4]:
            splits.append(4)


    splits = np.array(splits)
    gdf['split'] = splits
    subset = gdf

    

    min_examples_per_class = 0
    #There should be at least n examples of each class
    for i in range(5):
        classes, counts = np.unique(subset.loc[subset.split == i].driver, return_counts=True)
        if len(classes) < 12: #All classes must be present in each split 
            min_examples_per_class = 0 #Some class has 0 examples
            break
        if counts.min() > min_examples_per_class:
            min_examples_per_class = counts.min()
    
    if min_examples_per_class >= required_n_examples_per_class:
        finished = True
    

In [34]:
gdf['block'] = labels

In [36]:
for i in [0,1,2,3,4]:

    classes, counts = np.unique(subset.loc[subset.split == i].driver, return_counts=True)
    print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 104  146 1002  723  252 1261  437   54  780  289  174  896]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 269  112 1332  699  888  352 1031  342 1708  660  268 1244]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [3279   38 1010 1282 1696 1207 1626 1119 1413 1363 2108 1349]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1162   38  817  943 1601  834  981  574  537 1608  335  607]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 184   82  820 1339  552 1341  916 2896  

In [84]:
classes, counts = np.unique(subset.loc[subset.split == 0].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1787  200  946 1171  820  942 1343 1277  735  766  663 1200]


In [85]:
classes, counts = np.unique(subset.loc[subset.split == 1].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1658   69  926  900 1384 1427 1124  831  879  993 1390 1213]


In [86]:
classes, counts = np.unique(subset.loc[subset.split == 2].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 225   45 1322 1009  584  968  926 1043  488 1112  852  914]


In [87]:
classes, counts = np.unique(subset.loc[subset.split == 3].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 132   53  768  917 1334 1018  817  867 1362 1044  972  693]


In [88]:
classes, counts = np.unique(subset.loc[subset.split == 4].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1196   49 1019  989  867  640  781  967 1528 1078 1115  965]


In [47]:
labels = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')

In [48]:
labels['split_5'] = subset['split']

In [50]:
labels.to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

In [38]:
#pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)
subset.to_file('../data/tmp/labels_tmp2.gpkg')


In [40]:
labels_tmp2 = gpd.read_file('../data/tmp/labels_tmp2.gpkg')

In [42]:
labels_tmp2.split

0        4
1        1
2        1
3        4
4        4
        ..
55298    1
55299    1
55300    2
55301    1
55302    0
Name: split, Length: 55303, dtype: int64

In [52]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
#df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
#gdf = gpd.GeoDataFrame(df)


In [56]:
df['split'] = labels_tmp2.split

In [58]:
for i in [1, 2, 0, 3, 4]:
    print(np.unique(df.loc[df.split == i].driver, return_counts=True))

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([1658,   69,  926,  900, 1384, 1427, 1124,  831,  879,  993, 1390,
       1213]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([ 225,   45, 1322, 1009,  584,  968,  926, 1043,  488, 1112,  852,
        914]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([1787,  200,  946, 1171,  820,  942, 1343, 1277,  735,  766,  663,
       1200]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urba

In [61]:
df.to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

In [59]:
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df)


In [60]:
gdf.to_file('../data/tmp/tmp3.gpkg')

In [1]:
for i in [1, 2, 0, 3, 4]:
    print(np.unique(labels_tmp2.loc[labels_tmp2.split == i].driver, return_counts=True))

NameError: name 'np' is not defined

## Implementation of the block split myself

- this could be better because i can make adjustments to optimize the class ratios

Steps
- create grid
- do an intersection with the points gdf
- only keep grid cells containing points
- split cells into 5 groups
    - this can be random or I can e.g. split the cells into high-density cells and low-density cells and take this into account


### Add country

In [38]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df, geometry='geometry')
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_7301/1252657791.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [30]:
#world['geometry'] = world.to_crs(3857).geometry.buffer(100000, cap_style=3).to_crs(4326)


In [39]:
gdf['country'] = gdf.geometry.intersects(world).name

/tmp/ipykernel_7301/291178877.py:1: UserWarning: The indices of the two GeoSeries are different.
  gdf['country'] = gdf.geometry.intersects(world).name
/tmp/ipykernel_7301/291178877.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  gdf['country'] = gdf.geometry.intersects(world).name


In [40]:
gdf.crs = 4326

In [41]:
#This results in over 1.6k missing values because the world map is low resolution so many points don't intersect with any country
#gdf_with_countries = gdf.sjoin(world[['name', 'geometry']], how="left")
#gdf_with_countries['country'] = gdf_with_countries.name
#gdf_with_countries = gdf_with_countries.drop(['index_right', 'name'], axis=1)

In [64]:
from shapely.ops import nearest_points


def find_intersections_or_nearest(gdf_points, gdf_polygons):
    """
    For each point in gdf_points, find which polygon it intersects with. 
    If it doesn't intersect any polygon, find the closest polygon.
    
    Parameters:
    gdf_points (GeoDataFrame): A GeoDataFrame containing points.
    gdf_polygons (GeoDataFrame): A GeoDataFrame containing polygons.
    
    Returns:
    GeoDataFrame: A GeoDataFrame of points with a new column 'nearest_or_intersecting_polygon' indicating the index of the intersecting or nearest polygon.
    """
    # Initialize a column to store the index of the nearest or intersecting polygon
    gdf_points['country'] = None
    
    for point_index, point_row in gdf_points.iterrows():
        # Check for intersections first
        intersects = gdf_polygons.intersects(point_row.geometry)
        if intersects.any():
            # If there's an intersection, use the first intersecting polygon
            idx = intersects.idxmax()
            gdf_points.at[point_index, 'country'] = gdf_polygons.iloc[idx]['name']
        else:
            # Find the nearest polygon if no intersection
            # Calculate the nearest points between the current point and all polygons, then find the minimum
            distances = gdf_polygons.geometry.apply(lambda poly: point_row.geometry.distance(nearest_points(point_row.geometry, poly)[1]))
            nearest_polygon_index = distances.idxmin()
            gdf_points.at[point_index, 'country'] = gdf_polygons.iloc[nearest_polygon_index]['name']
    
    return gdf_points

In [70]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    gdf_with_countries.loc[gdf_with_countries['country'] == name, 'country'] = country_name_mapping[name]
    

In [66]:
gdf_with_countries = find_intersections_or_nearest(gdf, world)

In [73]:
pd.DataFrame(gdf_with_countries).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)